# Introduction


## References

* https://medium.com/@van3ssabandeira/o-famoso-spacy-90afb683b6fe
* [Python para PLN - Disciplinas USP](https://edisciplinas.usp.br/pluginfile.php/6305937/mod_resource/content/0/Aula%2010%20-%20python%20para%20PLN%20-%20spaCy.pdf)

# Requirements

In [1]:
# https://www.tensorflow.org/api_docs/python/tf/keras/layers

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import unicodedata
import datetime
import random
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Get Data

In [2]:
# transformando e abrindo o arquivo em formato csv
df = pd.read_excel('DateToTestSentiment_20210817.xls.xls')

# Data Prep Functions

* **preprocess_data** 
* **preprocess_text** 
* **tokenizacao**

In [3]:
def preprocess_data(data, 
                    columns,
                    null = True):
    
    df = data[columns]
    
    if null:
        df = df.dropna().reset_index().drop(columns=['index'])
    
    return df


def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True
                   ):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # corrects the bug that eliminates letters from words with accents
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    # removes special characters 
    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    #
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    # removes stopwords with less than two letters
    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # gets the number of rows and columns from the dataframe
    rows, cols = df.shape

    # creates the dataset column with the vectorized text
    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

# Pipeline Data Prep

  Tokeniza a coluna de comentários do dataset e depois remove as linhas sem texto

In [4]:
portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

#
df2 = preprocess_data(df, 
                      columns=['KEY','COMMENT_ID','COMMENT_TEXT'],
                      null = True
                     )

#
df2 = tokenizacao(df2)

# creates an intermediary dataframe
df3 = df2.copy()

# 
for n, item in enumerate(df2.token):
    if len(item)==0:
        df3 = df3.drop(n)

# Topicos de classificação

- Tópicos em ordem de prioridade que usaremos para a classificação
- Criação do dicionário que associa os labels a seu tópico
- Tokenização dos tópicos para usar na seleção dos labels

In [5]:
topics = ['AQUECIMENTO', 
          'ASSISTÊNCIA TÉCNICA', 
          'ATENDIMENTO', 
          'AUTO FALANTE', 
          'BATERIA', 
          'CAMERA', 
          'CARREGADOR', 
          'CUSTO BENEFICIO', 
          'DESIGN', 
          'ENTREGA', 
          'FLASH', 
          'FONE', 
          'JOGOS', 
          'MEMÓRIA',
          'PESO', 
          'PREÇO', 
          'PROCESSADOR', 
          'QUALIDADE', 
          'RESISTÊNCIA', 
          'TAMANHO', 
          'TELA', 
          'TRAVAMENTO',
          'VELOCIDADE', 
          'GENÉRICO/OUTRO'
         ]

#
dic_topics = {}
for n, item in enumerate(topics):
    dic_topics[n]=item

#
topics_token = [preprocess_text(item) for item in topics]

# Primeiros labels 

- Criação dos 23 primeiros labels, onde são decididos vendo se o comentário possui a palavra do label, em ordem de prioridade da lista

In [6]:
# defines empty list
lista_clusters = []

#
for n, objeto in enumerate(topics_token):
    
    if len(objeto) == 1:
        lista_1 = [item for item in df3.token if objeto[0] in item]
        for n, item in enumerate(lista_clusters):
            lista_1 = [item1 for item1 in lista_1 if item1 not in item]
        lista_clusters.append(lista_1)

    else:
        lista_2 = [item for item in df3.token if (
            objeto[0] in item) and (objeto[1] in item)]
        for n, item in enumerate(lista_clusters):
            lista_2 = [item1 for item1 in lista_2 if item1 not in item]
        lista_clusters.append(lista_2)

# Último Label

- Criação do último label, escolhendo os comentários no dataset que não possui nenhum dos 23 tópicos contidos no seu texto, sendo então caracterizado como "Genérico/outro"

- Temos uma lista onde cada elemento é uma lista com todas os comentários do dataset separados por seu label

- Como temos mais de 40 mil comentários na categoria "Genérico/outros", fizemos um balanceamento

- Ao final conferimos o tamanho dessa lista, onde cada um desses tamanho é a quantidade de comentários em seu respectivo label.

In [7]:
temp = []

#
for n, objeto in enumerate(lista_clusters):
    temp.extend(objeto)

#
ultimo_cluster = [item for item in df3.token if item not in temp]

#
lista_clusters[23] = ultimo_cluster

#
lista_clusters1 = lista_clusters.copy()

#
for n, item in enumerate(lista_clusters1):
    if len(item) > 3500:
        lista_clusters1[n] = random.sample(item, 2000)    

# Vectorization

 - definindo a função que usaremos para vetorizar os textos

In [8]:
#
nlp = spacy.load('pt_core_news_md')

#
def vec(s):
    return nlp.vocab[s].vector

# Vetorização - Coluna

Vetorizando toda a coluna de comentários e colocando em uma matriz final.

In [9]:
#
linhas = [len(item) for item in lista_clusters]

#
vec_size = 300

#
rows = sum(linhas)

#
list_of_matrix = [] 

#
final_feature_matrix = np.empty([rows, vec_size])

#
for n, item in enumerate(lista_clusters):
    for corpus in item: 
        matrix = np.empty([len(corpus), vec_size]) 
                                              
        for idx, word in enumerate(corpus):
            matrix[idx,:] = vec(word) 
        list_of_matrix.append(matrix)

#
for row in range(rows):
    final_feature_matrix[row,:] = list_of_matrix[row].mean(axis = 0)

# Labels e Shuffle

- Adicionando a matriz vetorizada, uma coluna com labels, baseada no tamanho de cada uma das listas contidas na "lista_clusters1"

- Fazendo um shuffle na matriz para diminuir as chances de viés no treinamento do modelo

In [10]:
labels = []

#
for n, item in enumerate(lista_clusters):
    for i, objeto in enumerate(item):
        labels.append(n)
    
#
x = np.array(labels)

#
x = x.reshape(-1,1)

#
final_matrix = np.concatenate((final_feature_matrix, x), axis=1)

#
final_matrix1 = shuffle(final_matrix, random_state = 42)

# Treinamento e predição

- Dividindo os dados em treino e teste

- Treinamos um knn com 2 vizinhos

- predição nos dados de teste

- conferindo acurácia

In [11]:
#
Xtreino, Xteste, ytreino, yteste = train_test_split(
    final_matrix1[:, 0:-1], final_matrix1[:, -1], train_size = 0.7, random_state=42)

#
knn = KNeighborsClassifier(n_neighbors = 2)

#
knn.fit(Xtreino, ytreino)

#
pred = knn.predict(Xteste)

#
accuracy_score(yteste,pred)

0.7767454350161117

# Testando o modelo em frase generica

- Testamos o modelo na frase genérica "otimo"

In [12]:
a = 'Otimo'

#
textoProcessado = preprocess_text(a)

#
matrix = np.empty([len(textoProcessado), 300])

#
for idx, word in enumerate(textoProcessado):
    matrix[idx,:] = vec(word)

#
final_feature_matrix = np.empty([1, 300])

#
final_feature_matrix = matrix.mean(axis = 0).reshape(1,-1)

knn.predict(final_feature_matrix)

dic_topics[knn.predict(final_feature_matrix)[0]]

'GENÉRICO/OUTRO'